# 10 - V2.2 Quick: Diagnostic ecart OOF/Kaggle + retraining cible + soumission

Ce notebook complete `07/08/09` en mode **decision de soumission**:
- diagnostic du gap OOF/Kaggle,
- retraining quick cible (4-6 runs),
- 2 soumissions (`robust`, `challenger`),
- rapport de decision.

## Clarification metrique (important)

- `~542 local` = score OOF interne (train/validation)
- `~1366 Kaggle public` = hidden test (1/3 public)
- non-comparabilite directe => le diagnostic doit separer:
  - `CV instability`,
  - `OOD/shift`,
  - `queue severity underfit`,
  - `calibration/post-processing`.


In [1]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
# Notebook is in notebooks/ds/, need to go to project root
while ROOT.name != "Calcul-prime-d-assurance" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.insurance_pricing.experiments.quick import gap_diagnosis as wf
import src.insurance_pricing.training as v2

SEED = 42
KAGGLE_PUBLIC_RMSE = 1366.0
MAX_QUICK_RUNS = 4
RUN_RETRAIN = True
RUN_SHAKEUP = True
N_SIM_SHAKEUP = 300
QUICK_FEATURE_SET_OVERRIDE = None
TE_COLS = ["code_postal", "cp3", "modele_vehicule", "marque_modele"]

ARTIFACT_QUICK = wf.ensure_quick_dir(ROOT)
print("ROOT:", ROOT)
print("ARTIFACT_QUICK:", ARTIFACT_QUICK)


ROOT: c:\Users\icemo\Downloads\Calcul-prime-d-assurance
ARTIFACT_QUICK: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_2_quick


In [2]:
# Chargement des artefacts existants (lecture seule)
ctx = wf.load_existing_artifacts(ROOT)
print("v1_run_registry:", ctx["v1_run_registry"].shape)
print("v2_run_registry:", ctx["v2_run_registry"].shape)
print("v2_oof:", ctx["v2_oof"].shape)
print("ds_metrics:", ctx["ds_metrics"].shape)


v1_run_registry: (11, 14)
v2_run_registry: (81, 18)
v2_oof: (8100000, 18)
ds_metrics: (1, 20)


## 2) Audit de comparabilite V1 / V2 / DS (bridge diagnostic)

Decision pratique:
- on mesure des signaux comparables (OOF, distributions, queue),
- on evite de confondre l'echelle OOF vs Kaggle public.


In [3]:
bridge_summary_df = wf.build_bridge_summary(ctx, kaggle_public_rmse=KAGGLE_PUBLIC_RMSE)
sub_dist_summary_df, sub_corr_df = wf.build_submission_distribution_summary(ctx)

display(bridge_summary_df)
display(sub_dist_summary_df)
display(sub_corr_df)


,model_version,run_id,rmse_primary_time,auc_freq,brier_freq,rmse_sev_pos,q99_ratio_pos,rmse_prime_top1pct,kaggle_public_rmse_user
0,v1_best_local,catboost|cb_c1|42|classic|none,542.672558,0.650693,0.054100,1476.722719,0.309563,NaN,NaN
1,v2_selected_top,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,543.897946,0.608750,0.054531,1462.342755,0.328733,NaN,NaN
2,ds_diagnostic_run,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,545.929482,0.594651,0.054613,1662.270082,0.328630,4548.227515,1366.0


,name,exists,n,mean,std,q90,q95,q99,max,share_zero
0,submission_v1,1,50000,97.606838,62.347925,172.878272,201.924632,284.707752,692.093891,0.0
1,submission_v2_robust,1,50000,83.892877,124.212128,174.953409,326.587276,632.933597,2258.126486,0.0
2,submission_v2_single,1,50000,83.892877,124.212128,174.953409,326.587276,632.933597,2258.126486,0.0


,left,right,corr_pred
0,v1,v2_robust,0.439401
1,v1,v2_single,0.439401
2,v2_robust,v2_single,1.000000


In [4]:
# Focus distribution V2 shortlist (si artefacts disponibles)
v2_sel_report = ctx["v2_selection_report"]
v2_pred_dist = ctx["v2_pred_dist"]
shortlist_ids = []
if len(v2_sel_report) and "run_id" in v2_sel_report.columns:
    sort_cols = [c for c in ["rank", "selection_score"] if c in v2_sel_report.columns]
    shortlist_ids = v2_sel_report.sort_values(sort_cols if sort_cols else v2_sel_report.columns.tolist()[:1])["run_id"].astype(str).head(10).tolist()

if len(v2_pred_dist) and shortlist_ids:
    focus = v2_pred_dist[v2_pred_dist["run_id"].astype(str).isin(shortlist_ids)].copy()
    cols = [c for c in [
        "run_id", "split", "sample", "pred_mean", "pred_std", "pred_q90", "pred_q99",
        "pred_q99_q90_ratio", "pred_identical_share", "pred_identical_share_nonzero", "distribution_collapse_flag"
    ] if c in focus.columns]
    display(focus[cols].sort_values(["run_id", "split", "sample"]).head(80))
else:
    print("Pred distribution audit V2 shortlist unavailable.")


,run_id,split,sample,pred_mean,pred_std,pred_q90,pred_q99,pred_q99_q90_ratio,pred_identical_share,distribution_collapse_flag
6,base_v2|catboost|two_part_classic|cb_v2_c1|42|...,aux_blocked5,oof,103.057424,53.980453,168.658469,232.089967,1.376094,0.02006,0
7,base_v2|catboost|two_part_classic|cb_v2_c1|42|...,aux_blocked5,test,104.531318,51.159845,168.033159,218.091684,1.297909,0.06278,0
8,base_v2|catboost|two_part_classic|cb_v2_c1|42|...,primary_time,oof,82.834039,58.523306,155.112245,190.263578,1.226619,0.20494,1
9,base_v2|catboost|two_part_classic|cb_v2_c1|42|...,primary_time,test,105.040754,40.952893,157.228630,184.050488,1.170591,0.05832,0
10,base_v2|catboost|two_part_classic|cb_v2_c1|42|...,secondary_group,oof,103.212540,61.538566,184.054017,255.817464,1.389904,0.02488,0
11,base_v2|catboost|two_part_classic|cb_v2_c1|42|...,secondary_group,test,103.669637,59.732800,184.867690,239.721823,1.296721,0.06830,0
30,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,aux_blocked5,oof,102.924291,55.634638,170.615725,226.214130,1.325869,0.01762,0
31,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,aux_blocked5,test,104.334133,53.354483,171.527474,223.035052,1.300288,0.05560,0
32,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,primary_time,oof,82.374855,59.723301,160.594941,191.994047,1.195517,0.20494,1
33,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,primary_time,test,104.447724,43.603389,159.920530,179.959079,1.125303,0.11716,0


## 3) Diagnostic "overfitting ou pas ?"

On produit un `gap_diagnosis_report` avec:
- stabilite multi-splits,
- queue (`q99_ratio_pos`, top1%`),
- mismatch OOF/test,
- flags heuristiques,
- classification de cause dominante.


In [5]:
gap_diagnosis_report, gap_diagnosis_summary = wf.build_gap_diagnosis(
    ctx,
    kaggle_public_rmse=KAGGLE_PUBLIC_RMSE,
    top_k=10,
)
wf.save_gap_diagnosis_artifacts(gap_diagnosis_report, gap_diagnosis_summary, out_dir=ARTIFACT_QUICK)

display(gap_diagnosis_report.head(10))
print("gap_diagnosis_summary:", gap_diagnosis_summary)


,run_id,feature_set,engine,family,config_id,seed,severity_mode,calibration,tail_mapper,rmse_aux_blocked5,...,q95_ratio_pos,q99_ratio_pos,auc_freq,brier_freq,tail_undercoverage_flag,distribution_mismatch_flag,cv_instability_flag,ood_risk_flag,split,kaggle_gap_hypothesis
0,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,base_v2,catboost,two_part_tweedie,cb_v2_c1,42,weighted_tail,isotonic,isotonic,542.595859,...,0.476975,0.328630,0.594651,0.054613,1,0,0,0,primary_time,Gap domine par sous-modelisation de la queue
1,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,base_v2,catboost,two_part_tweedie,cb_v2_c1,42,classic,isotonic,isotonic,542.571435,...,0.473715,0.318202,0.594651,0.054613,1,0,0,0,primary_time,Gap domine par sous-modelisation de la queue
2,base_v2|catboost|two_part_classic|cb_v2_c1|42|...,base_v2,catboost,two_part_classic,cb_v2_c1,42,classic,isotonic,isotonic,542.670668,...,0.456074,0.310143,0.594651,0.054613,1,0,0,0,primary_time,Gap domine par sous-modelisation de la queue
3,base_v2|xgboost|two_part_tweedie|xgb_v2_c1|42|...,base_v2,xgboost,two_part_tweedie,xgb_v2_c1,42,classic,none,isotonic,542.888025,...,0.442258,0.324916,0.585644,0.054694,1,0,0,0,primary_time,Gap domine par sous-modelisation de la queue
4,base_v2|lightgbm|two_part_classic|lgb_v2_c1|42...,base_v2,lightgbm,two_part_classic,lgb_v2_c1,42,classic,isotonic,isotonic,543.121771,...,0.440264,0.290314,0.577980,0.054724,1,0,0,0,primary_time,Gap domine par sous-modelisation de la queue
5,base_v2|xgboost|two_part_tweedie|xgb_v2_c1|42|...,base_v2,xgboost,two_part_tweedie,xgb_v2_c1,42,classic,isotonic,isotonic,542.883211,...,0.442258,0.324916,0.582529,0.054711,1,0,0,0,primary_time,Gap domine par sous-modelisation de la queue
6,base_v2|xgboost|two_part_classic|xgb_v2_c1|42|...,base_v2,xgboost,two_part_classic,xgb_v2_c1,42,classic,isotonic,isotonic,542.877905,...,0.459512,0.319072,0.582529,0.054711,1,0,0,0,primary_time,Gap domine par sous-modelisation de la queue
7,base_v2|xgboost|two_part_classic|xgb_v2_c1|42|...,base_v2,xgboost,two_part_classic,xgb_v2_c1,42,weighted_tail,isotonic,isotonic,542.842109,...,0.451664,0.291309,0.582529,0.054711,1,0,0,0,primary_time,Gap domine par sous-modelisation de la queue
8,base_v2|xgboost|two_part_classic|xgb_v2_c1|42|...,base_v2,xgboost,two_part_classic,xgb_v2_c1,42,weighted_tail,none,isotonic,542.854402,...,0.451664,0.291309,0.585644,0.054694,1,0,0,0,primary_time,Gap domine par sous-modelisation de la queue
9,base_v2|xgboost|two_part_classic|xgb_v2_c1|42|...,base_v2,xgboost,two_part_classic,xgb_v2_c1,42,classic,none,isotonic,542.886507,...,0.459512,0.319072,0.585644,0.054694,1,0,0,0,primary_time,Gap domine par sous-modelisation de la queue


gap_diagnosis_summary: {'kaggle_public_rmse_user': 1366.0, 'note_metric_non_comparable': True, 'n_runs_analyzed': 10, 'global_unseen_focus_flag': 0, 'dominant_hypothesis_top_run': 'Gap domine par sous-modelisation de la queue', 'dominant_hypothesis_counts': {'Gap domine par sous-modelisation de la queue': 10}}


**A completer apres execution**

- Si les gaps multi-splits sont faibles / negatifs mais que `q99_ratio_pos` est bas, le gap Kaggle est plus probablement **queue + OOD** que surapprentissage CV.
- Si un run explose `primary_time` mais degrade `secondary/aux`, le signal d'overfitting CV devient credible.


## 4) Strategie quick (1-2h)

- `MAX_QUICK_RUNS=4` par defaut (4-6 possible)
- shortlist data-driven depuis `selection_report_v2.csv`
- pas de refactor pipeline / pas de nouvelles features lourdes


In [6]:
run_candidates_screen = wf.select_quick_candidates(
    ctx["v2_selection_report"],
    ctx["v2_run_registry"],
    max_runs=MAX_QUICK_RUNS,
)
if len(run_candidates_screen):
    run_candidates_screen.to_csv(ARTIFACT_QUICK / "run_candidates_screen.csv", index=False)
    print("saved:", ARTIFACT_QUICK / "run_candidates_screen.csv")
    display(run_candidates_screen)
else:
    print("No candidates found; diagnostic-only mode.")


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_2_quick\run_candidates_screen.csv


,run_id,feature_set,engine,family,config_id,seed,severity_mode,calibration,tail_mapper,rmse_aux_blocked5,...,accepted_secondary,accepted_aux,accepted_tail,accepted_collapse,accepted_dispersion,accepted,decision_reason,rank,candidate_rank_quick,candidate_source
0,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,base_v2,catboost,two_part_tweedie,cb_v2_c1,42,weighted_tail,isotonic,isotonic,542.595859,...,True,True,True,True,True,True,accepted,1,1,selection_report_v2
1,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,base_v2,catboost,two_part_tweedie,cb_v2_c1,42,classic,isotonic,isotonic,542.571435,...,True,True,True,True,True,True,accepted,2,2,selection_report_v2
2,base_v2|catboost|two_part_classic|cb_v2_c1|42|...,base_v2,catboost,two_part_classic,cb_v2_c1,42,classic,isotonic,isotonic,542.670668,...,True,True,True,True,True,True,accepted,3,3,selection_report_v2
3,base_v2|xgboost|two_part_tweedie|xgb_v2_c1|42|...,base_v2,xgboost,two_part_tweedie,xgb_v2_c1,42,classic,none,isotonic,542.888025,...,True,True,True,True,True,True,accepted,4,4,selection_report_v2


In [7]:
# 5) Retraining quick cible (benchmark)
quick_result = {
    "fold_df": pd.DataFrame(),
    "run_df": pd.DataFrame(),
    "pred_df": pd.DataFrame(),
    "dist_df": pd.DataFrame(),
    "errors_df": pd.DataFrame(),
    "train_raw": pd.DataFrame(),
    "test_raw": pd.DataFrame(),
    "feature_sets": {},
}

if RUN_RETRAIN and len(run_candidates_screen):
    quick_result = wf.run_quick_benchmark(
        data_dir=ctx["data_dir"],
        candidates_df=run_candidates_screen,
        te_cols=TE_COLS,
        seed=SEED,
        feature_set_override=QUICK_FEATURE_SET_OVERRIDE,
        out_dir=ARTIFACT_QUICK,
    )
    print("quick run_df:", quick_result["run_df"].shape)
    print("quick pred_df:", quick_result["pred_df"].shape)
    print("quick dist_df:", quick_result["dist_df"].shape)
    if len(quick_result["errors_df"]):
        display(quick_result["errors_df"])
else:
    print("Retraining skipped (RUN_RETRAIN=False or no candidates).")


quick run_df: (36, 27)
quick pred_df: (3600000, 19)
quick dist_df: (72, 25)


In [8]:
# 5bis) Scoring quick + selection robuste/challenger
retrain_registry_quick = pd.DataFrame()
selected_robust_quick = None
selected_challenger_quick = None

if len(quick_result["run_df"]):
    retrain_registry_quick = wf.score_quick_runs(
        quick_result["run_df"],
        quick_result["pred_df"],
        quick_result["dist_df"],
        seed=SEED,
        n_sim_shakeup=N_SIM_SHAKEUP,
        run_shakeup=RUN_SHAKEUP,
    )
    robust_row, challenger_row = wf.choose_robust_and_challenger(retrain_registry_quick)
    selected_robust_quick = robust_row
    selected_challenger_quick = challenger_row

    retrain_registry_quick["selection_status"] = "rejected"
    if robust_row:
        retrain_registry_quick.loc[
            retrain_registry_quick["run_id"].astype(str) == str(robust_row["run_id"]),
            "selection_status",
        ] = "selected_robust"
    if challenger_row:
        mask_ch = retrain_registry_quick["run_id"].astype(str) == str(challenger_row["run_id"])
        retrain_registry_quick.loc[mask_ch, "selection_status"] = np.where(
            retrain_registry_quick.loc[mask_ch, "selection_status"].eq("selected_robust"),
            "selected_robust",
            "selected_challenger",
        )

    retrain_registry_quick.to_csv(ARTIFACT_QUICK / "retrain_registry_quick.csv", index=False)
    print("saved:", ARTIFACT_QUICK / "retrain_registry_quick.csv")
    display(retrain_registry_quick.sort_values(["selection_score_quick", "rmse_primary_time"]).head(20))
else:
    print("No quick runs to score.")


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_2_quick\retrain_registry_quick.csv


,run_id,feature_set,engine,family,tweedie_power,config_id,seed,severity_mode,calibration,tail_mapper,...,distribution_alignment_score,distribution_collapse_flag,shakeup_std_gap,tail_penalty,rmse_gap_secondary_pos,rmse_gap_aux_pos,shakeup_penalty,selection_score_quick,passes_local_guardrails,selection_status
9,robust_v2|catboost|two_part_tweedie|1.5|cb_v2_...,robust_v2,catboost,two_part_tweedie,1.5,cb_v2_c1,42,classic,isotonic,piecewise_monotone_safe,...,-1.918068,0.0,37.021496,0.027938,0.000000,4.303696,37.021496,594.507678,False,selected_robust
5,compact_v2|catboost|two_part_tweedie|1.5|cb_v2...,compact_v2,catboost,two_part_tweedie,1.5,cb_v2_c1,42,classic,isotonic,piecewise_monotone_safe,...,-2.417493,0.0,36.892926,0.193026,0.000000,1.134490,36.892926,596.052712,False,rejected
4,compact_v2|catboost|two_part_classic|1.5|cb_v2...,compact_v2,catboost,two_part_classic,1.5,cb_v2_c1,42,classic,isotonic,piecewise_monotone_safe,...,-1.895863,0.0,36.942704,1.285222,1.155879,0.698932,36.942704,596.469087,False,rejected
7,compact_v2|xgboost|two_part_tweedie|1.5|xgb_v2...,compact_v2,xgboost,two_part_tweedie,1.5,xgb_v2_c1,42,classic,none,piecewise_monotone_safe,...,-3.068211,0.0,37.052011,1.677438,0.000000,0.000000,37.052011,598.385256,True,rejected
3,base_v2|xgboost|two_part_tweedie|1.5|xgb_v2_c1...,base_v2,xgboost,two_part_tweedie,1.5,xgb_v2_c1,42,classic,none,piecewise_monotone_safe,...,-2.356917,0.0,37.254764,0.233684,7.244595,3.581697,37.254764,598.652424,False,selected_challenger
0,base_v2|catboost|two_part_classic|1.5|cb_v2_c1...,base_v2,catboost,two_part_classic,1.5,cb_v2_c1,42,classic,isotonic,piecewise_monotone_safe,...,-2.011209,0.0,37.042541,0.544743,2.628622,3.878174,37.042541,598.983292,False,rejected
11,robust_v2|xgboost|two_part_tweedie|1.5|xgb_v2_...,robust_v2,xgboost,two_part_tweedie,1.5,xgb_v2_c1,42,classic,none,piecewise_monotone_safe,...,-3.170486,0.0,37.015553,4.517025,0.000000,0.000000,37.015553,599.684704,True,rejected
1,base_v2|catboost|two_part_tweedie|1.5|cb_v2_c1...,base_v2,catboost,two_part_tweedie,1.5,cb_v2_c1,42,classic,isotonic,piecewise_monotone_safe,...,-2.371911,0.0,36.980683,1.554420,1.517288,0.000000,36.980683,599.735953,False,rejected
8,robust_v2|catboost|two_part_classic|1.5|cb_v2_...,robust_v2,catboost,two_part_classic,1.5,cb_v2_c1,42,classic,isotonic,piecewise_monotone_safe,...,-2.120824,0.0,36.976011,2.874833,6.620165,2.887835,36.976011,601.816802,False,rejected
6,compact_v2|catboost|two_part_tweedie|1.5|cb_v2...,compact_v2,catboost,two_part_tweedie,1.5,cb_v2_c1,42,weighted_tail,isotonic,piecewise_monotone_safe,...,-3.853488,0.0,36.708670,3.313505,4.284645,0.000000,36.708670,606.709624,False,rejected


In [9]:
# 6) Refit 100% train + 2 soumissions (robust/challenger)
submission_result = wf.build_quick_submissions(
    ctx,
    quick_result=quick_result,
    scored_df=retrain_registry_quick,
    robust_row=selected_robust_quick,
    challenger_row=selected_challenger_quick,
    te_cols=TE_COLS,
    seed=SEED,
    out_dir=ARTIFACT_QUICK,
)

print("robust meta:", submission_result["robust_meta"])
print("challenger meta:", submission_result["challenger_meta"])
display(pd.DataFrame([
    {"submission": "robust", **submission_result["robust_meta"]},
    {"submission": "challenger", **submission_result["challenger_meta"]},
]))


robust meta: {'run_id': 'robust_v2|catboost|two_part_tweedie|1.5|cb_v2_c1|42|classic|isotonic|piecewise_monotone_safe', 'source': 'fit_full_predict_fulltrain'}
challenger meta: {'run_id': 'base_v2|xgboost|two_part_tweedie|1.5|xgb_v2_c1|42|classic|none|piecewise_monotone_safe', 'source': 'fit_full_predict_fulltrain'}


,submission,run_id,source
0,robust,robust_v2|catboost|two_part_tweedie|1.5|cb_v2_...,fit_full_predict_fulltrain
1,challenger,base_v2|xgboost|two_part_tweedie|1.5|xgb_v2_c1...,fit_full_predict_fulltrain


In [10]:
# 7) Anti-fausse amelioration: checks, audits distribution, shake-up final
quick_checks = wf.build_quick_checks(
    scored_df=retrain_registry_quick,
    quick_pred_df=quick_result["pred_df"],
    robust_meta=submission_result["robust_meta"],
    challenger_meta=submission_result["challenger_meta"],
    robust_submission=submission_result["robust_submission"],
    challenger_submission=submission_result["challenger_submission"],
    baseline_row_v2=submission_result["baseline_v2_selected_row"],
    seed=SEED,
    n_sim_shakeup=N_SIM_SHAKEUP,
    out_dir=ARTIFACT_QUICK,
)

quick_checks_df = quick_checks["checks_df"]
submission_dist_df = quick_checks["submission_dist_df"]

# Merge/overwrite pred distribution audit quick (runs + submissions) sans toucher a V2
pred_distribution_audit_quick_combined = quick_result["dist_df"].copy() if len(quick_result["dist_df"]) else pd.DataFrame()
if len(submission_dist_df):
    pred_distribution_audit_quick_combined = pd.concat([pred_distribution_audit_quick_combined, submission_dist_df], ignore_index=True, sort=False)
if len(pred_distribution_audit_quick_combined):
    pred_distribution_audit_quick_combined.to_csv(ARTIFACT_QUICK / "pred_distribution_audit_v2_2_quick.csv", index=False)
    print("saved:", ARTIFACT_QUICK / "pred_distribution_audit_v2_2_quick.csv")

display(quick_checks_df)
display(submission_dist_df)
if len(quick_checks["shakeup_robust"]):
    print("robust shakeup std gap:", float(quick_checks["shakeup_robust"]["gap_public_minus_private"].std(ddof=0)))
if len(quick_checks["shakeup_challenger"]):
    print("challenger shakeup std gap:", float(quick_checks["shakeup_challenger"]["gap_public_minus_private"].std(ddof=0)))


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_2_quick\pred_distribution_audit_v2_2_quick.csv


,submission,run_id,rmse_primary_time,rmse_secondary_group,rmse_aux_blocked5,q99_ratio_pos,rmse_prime_top1pct,distribution_collapse_flag,delta_rmse_primary_vs_baseline,delta_q99_ratio_vs_baseline,passes_rmse_tol,passes_q99_tol,passes_secondary_tol,passes_aux_tol,passes_collapse_flag,local_overall_pass
0,robust,robust_v2|catboost|two_part_tweedie|1.5|cb_v2_...,553.388328,550.834467,557.692024,0.998603,4532.662229,0.0,9.490383,0.669870,False,True,True,False,True,False
1,challenger,base_v2|xgboost|two_part_tweedie|1.5|xgb_v2_c1...,553.393912,560.638507,556.975609,0.988316,4545.784708,0.0,9.495966,0.659583,False,True,False,False,True,False


,submission_name,run_id,split,sample,n,pred_mean,pred_std,pred_q50,pred_q90,pred_q95,...,pred_share_nonzero,pred_identical_share,pred_identical_share_nonzero,pred_q99_q90_ratio,pred_n_nonzero,pred_q90_nonzero,pred_q99_nonzero,pred_q99_q90_ratio_nonzero,collapse_use_nonzero_support,distribution_collapse_flag
0,submission_v2_2_quick_robust,submission_v2_2_quick_robust,test,test,50000,80.361114,51.777235,73.014429,151.844838,171.282026,...,1.0,0.00004,0.00004,1.405041,50000,151.844838,213.348234,1.405041,0,0
1,submission_v2_2_quick_challenger,submission_v2_2_quick_challenger,test,test,50000,49.871307,92.254298,18.933201,128.711023,201.639664,...,1.0,0.00004,0.00004,3.394221,50000,128.711023,436.873694,3.394221,0,0


robust shakeup std gap: 37.021495832884185
challenger shakeup std gap: 37.254764456335785


In [11]:
# 7bis) OOF compare V1 / V2 / V2.2
oof_compare_v1_v2_v22 = wf.build_oof_compare_artifact(
    ctx,
    quick_pred_df=quick_result["pred_df"],
    robust_run_id=submission_result["robust_meta"].get("run_id"),
    challenger_run_id=submission_result["challenger_meta"].get("run_id"),
    out_dir=ARTIFACT_QUICK,
)
if len(oof_compare_v1_v2_v22):
    print("saved:", ARTIFACT_QUICK / "oof_compare_v1_v2_v22.parquet", oof_compare_v1_v2_v22.shape)
    display(oof_compare_v1_v2_v22.head(10))
else:
    print("OOF compare not generated (insufficient sources).")


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_2_quick\oof_compare_v1_v2_v22.parquet (50000, 6)


,row_idx,y_true,pred_v1_best,pred_v2_selected,pred_v22_quick_robust,pred_v22_quick_challenger
0,0,0.00,NaN,0.0,0.0,0.0
1,1,0.00,NaN,0.0,0.0,0.0
2,2,0.00,NaN,0.0,0.0,0.0
3,3,0.00,NaN,0.0,0.0,0.0
4,4,0.00,NaN,0.0,0.0,0.0
5,5,0.00,NaN,0.0,0.0,0.0
6,6,0.00,NaN,0.0,0.0,0.0
7,7,0.00,NaN,0.0,0.0,0.0
8,8,927.16,NaN,0.0,0.0,0.0
9,9,0.00,NaN,0.0,0.0,0.0


In [12]:
# 8) Rapport de decision de soumission
decision_path = wf.write_submission_decision_report(
    ctx,
    kaggle_public_rmse=KAGGLE_PUBLIC_RMSE,
    gap_summary=gap_diagnosis_summary,
    checks_df=quick_checks_df if "quick_checks_df" in globals() else pd.DataFrame(),
    scored_df=retrain_registry_quick,
    robust_meta=submission_result["robust_meta"],
    challenger_meta=submission_result["challenger_meta"],
    baseline_v2_row=submission_result["baseline_v2_selected_row"],
    out_dir=ARTIFACT_QUICK,
)
print("saved:", decision_path)
print(decision_path.read_text(encoding="utf-8")[:3000])


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_2_quick\submission_decision_v2_2_quick.md
# Submission decision V2.2 Quick

## 1) Contexte
- Kaggle public (utilisateur): ~1366.000
- OOF local vs Kaggle public: metriques non directement comparables.

## 2) Diagnostic du gap (resume)
- Hypothese dominante (top run): Gap domine par sous-modelisation de la queue
- OOD focus flag (categories fines): 0
- Nombre de runs analyses: 10

## 3) Tableau comparatif (local)

| variant               | run_id                                                                                       |   rmse_primary_time |   q99_ratio_pos |   selection_score_quick |
|:----------------------|:---------------------------------------------------------------------------------------------|--------------------:|----------------:|------------------------:|
| V1_best_local_ref     | catboost|cb_c1|42|classic|none                                                               |             542.67

## Fin de notebook - checklist

Artefacts attendus (quick):
- `run_candidates_screen.csv`
- `gap_diagnosis_report.csv` + `gap_diagnosis_summary.json`
- `retrain_registry_quick.csv`
- `pred_distribution_audit_v2_2_quick.csv`
- `submission_v2_2_quick_robust.csv`
- `submission_v2_2_quick_challenger.csv`
- `submission_decision_v2_2_quick.md`
- `oof_compare_v1_v2_v22.parquet`
- `shakeup_quick_robust.parquet` / `shakeup_quick_challenger.parquet` (si `RUN_SHAKEUP=True`)
